# LabMiM — Treinamento solrad_correction no Google Colab (GPU)

Treina os modelos de correção de radiação (SVM / LSTM / Transformer) usando o CLI
`solrad-colab`, com artefatos persistidos no seu Google Drive (checkpoints `best.pt`/`last.pt`,
métricas, predições com timestamps e logs do TensorBoard).

**Runtime → Change runtime type → GPU (T4)** antes de executar.
Guia completo: `notebooks/README.md` · Documentação: `docs/solrad_correction.md`.

In [ ]:
!nvidia-smi -L
%pip install -q "labmim-micrometeorology[tcc] @ git+https://github.com/Bruno-Mascarenhas/micrometeorology.git"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Layout recomendado no Drive: MyDrive/labmim/{data,runs}
DATA = "/content/drive/MyDrive/labmim/data"  # coloque aqui os .dat / parquet
RUNS = "/content/drive/MyDrive/labmim/runs/solrad"  # best.pt, last.pt, métricas, tensorboard

In [ ]:
# Baixa os configs de experimento do repositório (ou aponte para os seus no Drive)
!git clone -q --depth 1 https://github.com/Bruno-Mascarenhas/micrometeorology.git /content/repo
CONFIG = "/content/repo/configs/tcc/experiments/lstm_hourly.yaml"
# Ajuste data.path dentro do YAML para os arquivos em {DATA} antes de treinar, se necessário.

In [ ]:
# Métricas ao vivo (o diretório é criado pelo treino)
%load_ext tensorboard
%tensorboard --logdir {RUNS}

In [ ]:
# Treino na GPU com saída no Drive. --output-dir sobrepõe experiment.output_dir do YAML,
# então tudo (checkpoints, metrics.json, predictions.csv com timestamps) sobrevive à sessão.
!solrad-colab --config {CONFIG} --output-dir {RUNS} --device cuda --amp

## Retomando um treino interrompido

`model.max_epochs` é o orçamento TOTAL: retomar treina apenas as épocas restantes e o
`best.pt` anterior nunca é sobrescrito por uma época pior. Para estender um treino já
concluído, aumente `max_epochs` no YAML.

In [ ]:
import glob

last = sorted(glob.glob(f"{RUNS}/**/checkpoints/last.pt", recursive=True))
print("checkpoints:", last)
if last:
    get_ipython().system(
        f"solrad-colab --config {CONFIG} --output-dir {RUNS} --device cuda --amp --resume {last[-1]}"
    )

## Onde ficam os artefatos

```
{RUNS}/<experiment-name>/
├── checkpoints/best.pt · last.pt   # retomáveis; state_dict simples (funciona com --compile)
├── models/                          # modelo final + preprocessing (salvos logo após o fit)
├── metrics.json · config_resolved.json · metadata.json
└── predictions/predictions.csv     # sempre com timestamps
```
As predições em GPU são float32 — as métricas batem com uma avaliação em CPU.